# Ficheros · Compresión, configuración y ficheros grandes

Ejercicios con la biblioteca estándar para tres situaciones habituales: comprimir datos con `gzip`, `bz2` y `lzma` (y leerlos sin descomprimirlos enteros), leer configuración en TOML con `tomllib` y en INI con `configparser`, y trabajar con ficheros grandes sin cargarlos en memoria: `mmap`, lectura por bloques y por líneas con generadores y una ordenación externa con `heapq.merge`. Los ficheros temporales usan `tempfile`. Todos los ficheros se crean junto al notebook.

## 1. Generar un registro de medidas

Generar con una semilla fija un registro de 60 000 líneas de una red de estaciones meteorológicas de Canarias, con el formato `fecha;estación;temperatura;humedad`, y guardarlo en `medidas.csv`. Es texto muy repetitivo, ideal para comprimir. Mostrar las primeras líneas y el tamaño.

In [1]:
import random
from datetime import datetime, timedelta
from pathlib import Path

ESTACIONES = {"Izaña": 9.5, "Tenerife Norte": 17.0, "Gando": 21.5, "Fuerteventura": 21.0, "La Palma": 20.0, "Hierro": 20.5}
TEXTO = Path("medidas.csv")

rng = random.Random(2024)
inicio = datetime(2024, 1, 1)
lineas = []
for i in range(60_000):
    estacion = rng.choice(list(ESTACIONES))
    momento = inicio + timedelta(minutes=10 * (i // len(ESTACIONES)))
    temperatura = rng.gauss(ESTACIONES[estacion], 3)
    lineas.append(f"{momento:%Y-%m-%dT%H:%M};{estacion};{temperatura:.1f};{rng.randint(30, 95)}\n")
TEXTO.write_text("".join(lineas), encoding="utf-8", newline="\n")
datos = TEXTO.read_bytes()

print("".join(lineas[:4]), end="")
print(f"Líneas: {len(lineas)} | bytes: {len(datos):,}")
assert datos.count(b"\n") == 60_000

2024-01-01T00:00;Fuerteventura;22.6;55
2024-01-01T00:00;Hierro;24.1;82
2024-01-01T00:00;Hierro;20.7;93
2024-01-01T00:00;Gando;17.8;83
Líneas: 60000 | bytes: 2,073,896


## 2. Comprimir con `gzip`, `bz2` y `lzma`

Comprimir el contenido de `medidas.csv` con los tres módulos y comparar el tamaño y el ratio de compresión en una tabla. Guardar los tres ficheros comprimidos (`.gz`, `.bz2` y `.xz`) con su función `open` y comprobar que al descomprimirlos se recuperan exactamente los mismos bytes.

In [2]:
import bz2
import gzip
import lzma

modulos = {"gzip": (gzip, ".gz"), "bz2": (bz2, ".bz2"), "lzma": (lzma, ".xz")}

print(f"{'módulo':<6} {'bytes':>10} {'ratio':>7} {'ahorro':>7}")
for nombre, (modulo, ext) in modulos.items():
    comprimido = modulo.compress(datos)
    ratio = len(datos) / len(comprimido)
    print(f"{nombre:<6} {len(comprimido):>10,} {ratio:>7.2f} {1 - 1 / ratio:>7.1%}")
    with modulo.open(f"medidas.csv{ext}", "wb") as f:
        f.write(datos)
    assert modulo.decompress(comprimido) == datos
    with modulo.open(f"medidas.csv{ext}", "rb") as f:
        assert f.read() == datos

módulo      bytes   ratio  ahorro


gzip      325,134    6.38   84.3%


bz2       226,347    9.16   89.1%


lzma      249,084    8.33   88.0%


## 3. Leer un fichero comprimido en *streaming*

Calcular la temperatura media de cada estación leyendo `medidas.csv.gz` línea a línea con `gzip.open(..., "rt")`, sin descomprimirlo entero en memoria. Repetir con `.bz2` y `.xz` (la misma función sirve para los tres si se elige el módulo según la extensión) y comprobar que los tres dan lo mismo que el fichero sin comprimir.

In [3]:
from collections import defaultdict

ABRIR = {".gz": gzip.open, ".bz2": bz2.open, ".xz": lzma.open, ".csv": open}


def medias_por_estacion(ruta):
    suma, cuenta = defaultdict(float), defaultdict(int)
    with ABRIR[Path(ruta).suffix](ruta, "rt", encoding="utf-8") as f:
        for linea in f:
            _, estacion, temperatura, _ = linea.rstrip("\n").split(";")
            suma[estacion] += float(temperatura)
            cuenta[estacion] += 1
    return {e: round(suma[e] / cuenta[e], 2) for e in sorted(suma)}


medias = medias_por_estacion("medidas.csv.gz")
for estacion, media in medias.items():
    print(f"{estacion:<15} {media:6.2f} °C")
for ruta in ["medidas.csv.bz2", "medidas.csv.xz", "medidas.csv"]:
    assert medias_por_estacion(ruta) == medias

Fuerteventura    20.99 °C
Gando            21.49 °C
Hierro           20.52 °C
Izaña             9.49 °C
La Palma         20.00 °C
Tenerife Norte   16.95 °C


## 4. Compresión incremental

Cuando los datos llegan poco a poco no se pueden comprimir de una vez. Comprimir `medidas.csv` en bloques de 8 KiB con un `bz2.BZ2Compressor` y un `lzma.LZMACompressor` (llamando a `compress` con cada bloque y a `flush` al final), y descomprimir con sus `Decompressor` también por bloques. Comprobar que el resultado coincide con la compresión de una sola vez.

In [4]:
def comprimir_por_bloques(compresor, datos, tamaño=8192):
    partes = [compresor.compress(datos[i:i + tamaño]) for i in range(0, len(datos), tamaño)]
    partes.append(compresor.flush())
    return b"".join(partes)


def descomprimir_por_bloques(descompresor, datos, tamaño=8192):
    return b"".join(descompresor.decompress(datos[i:i + tamaño]) for i in range(0, len(datos), tamaño))


for nombre, compresor, descompresor, de_golpe in [
    ("bz2", bz2.BZ2Compressor(), bz2.BZ2Decompressor(), bz2.compress),
    ("lzma", lzma.LZMACompressor(), lzma.LZMADecompressor(), lzma.compress),
]:
    comprimido = comprimir_por_bloques(compresor, datos)
    print(f"{nombre:<5} por bloques: {len(comprimido):,} bytes | igual que de golpe: {comprimido == de_golpe(datos)}")
    assert descomprimir_por_bloques(descompresor, comprimido) == datos

bz2   por bloques: 226,347 bytes | igual que de golpe: True


lzma  por bloques: 249,084 bytes | igual que de golpe: True


## 5. `tempfile`: directorios y ficheros temporales

Dentro de un `tempfile.TemporaryDirectory` (creado junto al notebook con `dir="."`), escribir varios ficheros de trabajo y comprobar que el directorio desaparece al salir del `with`. Después, usar un `NamedTemporaryFile` con `delete_on_close=False` para escribir un fichero, cerrarlo, volver a abrirlo por su nombre (en Windows no se puede abrir dos veces mientras está abierto) y comprobar que se borra al salir del bloque. No imprimir las rutas, que cambian en cada ejecución.

In [5]:
import tempfile

with tempfile.TemporaryDirectory(dir=".", prefix="trabajo_") as carpeta:
    carpeta = Path(carpeta)
    for isla in ["tenerife", "gran_canaria", "lanzarote"]:
        (carpeta / f"{isla}.txt").write_text(f"datos de {isla}\n", encoding="utf-8")
    print("Ficheros dentro:", sorted(p.name for p in carpeta.iterdir()))
print("¿Existe al salir?", carpeta.exists())
assert not carpeta.exists()

with tempfile.NamedTemporaryFile("w", suffix=".csv", dir=".", delete_on_close=False, encoding="utf-8") as tmp:
    tmp.write("isla;habitantes\nLa Graciosa;750\n")
    tmp.close()
    contenido = Path(tmp.name).read_text(encoding="utf-8")
    nombre = Path(tmp.name)
    print("Releído por su nombre:", contenido.splitlines())
print("¿Existe al salir?", nombre.exists())
assert not nombre.exists() and contenido.startswith("isla;")

Ficheros dentro: ['gran_canaria.txt', 'lanzarote.txt', 'tenerife.txt']
¿Existe al salir? False
Releído por su nombre: ['isla;habitantes', 'La Graciosa;750']
¿Existe al salir? False


## 6. Leer configuración TOML con `tomllib`

Escribir el fichero `config.toml` de una aplicación de reservas de guaguas con tablas, una lista de tablas (`[[rutas]]`), fechas y listas. Leerlo con `tomllib` (en modo binario) y mostrar los tipos de Python que se obtienen. Comprobar que un TOML mal formado lanza `tomllib.TOMLDecodeError` y mostrar el mensaje.

In [6]:
import tomllib

Path("config.toml").write_text('''\
nombre = "Reservas Guaguas"
version = "2.1.0"
depuracion = false

[servidor]
host = "127.0.0.1"
puerto = 8080
tiempo_maximo = 2.5

[temporada]
inicio = 2025-06-21
fin = 2025-09-22
islas = ["Tenerife", "Gran Canaria", "La Palma"]

[[rutas]]
linea = 110
origen = "Santa Cruz"
destino = "Costa Adeje"

[[rutas]]
linea = 30
origen = "Las Palmas"
destino = "Maspalomas"
''', encoding="utf-8")

with open("config.toml", "rb") as f:
    config = tomllib.load(f)

for clave in ["nombre", "depuracion"]:
    print(f"{clave:<14} {config[clave]!r:<22} {type(config[clave]).__name__}")
for clave, valor in config["servidor"].items():
    print(f"{clave:<14} {valor!r:<22} {type(valor).__name__}")
print("Temporada:", config["temporada"]["inicio"], "->", config["temporada"]["fin"],
      f"({(config['temporada']['fin'] - config['temporada']['inicio']).days} días)")
for ruta in config["rutas"]:
    print(f"  Línea {ruta['linea']:>3}: {ruta['origen']} -> {ruta['destino']}")
assert config["servidor"]["puerto"] == 8080 and len(config["rutas"]) == 2

try:
    tomllib.loads('puerto = 80\npuerto = 81\n')
except tomllib.TOMLDecodeError as e:
    print("TOMLDecodeError:", e)

nombre         'Reservas Guaguas'     str
depuracion     False                  bool
host           '127.0.0.1'            str
puerto         8080                   int
tiempo_maximo  2.5                    float
Temporada: 2025-06-21 -> 2025-09-22 (93 días)
  Línea 110: Santa Cruz -> Costa Adeje
  Línea  30: Las Palmas -> Maspalomas
TOMLDecodeError: Cannot overwrite a value (at line 2, column 12)


## 7. Configuración con valores por defecto

Escribir `cargar_config(texto)` que combine unos valores por defecto con lo que venga en el TOML: las tablas se fusionan clave a clave (recursivamente) y las claves desconocidas se rechazan con un `ValueError`. Probarla con una configuración parcial y con otra que tenga una errata.

In [7]:
POR_DEFECTO = {"depuracion": False, "servidor": {"host": "localhost", "puerto": 8000, "tiempo_maximo": 5.0},
               "registro": {"nivel": "INFO", "fichero": "app.log"}}


def fusionar(base, cambios, ruta=""):
    resultado = dict(base)
    for clave, valor in cambios.items():
        if clave not in base:
            raise ValueError(f"clave desconocida: {ruta}{clave}")
        if isinstance(base[clave], dict):
            resultado[clave] = fusionar(base[clave], valor, f"{ruta}{clave}.")
        else:
            resultado[clave] = valor
    return resultado


def cargar_config(texto):
    return fusionar(POR_DEFECTO, tomllib.loads(texto))


config = cargar_config('[servidor]\npuerto = 9090\n[registro]\nnivel = "DEBUG"\n')
print(config)
assert config["servidor"] == {"host": "localhost", "puerto": 9090, "tiempo_maximo": 5.0}
assert POR_DEFECTO["servidor"]["puerto"] == 8000

try:
    cargar_config('[servidor]\npuerta = 9090\n')
except ValueError as e:
    print("Error:", e)

{'depuracion': False, 'servidor': {'host': 'localhost', 'puerto': 9090, 'tiempo_maximo': 5.0}, 'registro': {'nivel': 'DEBUG', 'fichero': 'app.log'}}
Error: clave desconocida: servidor.puerta


## 8. Ficheros INI con `configparser`

Escribir `tiendas.ini` con una sección `[DEFAULT]` cuyos valores heredan todas las secciones, e interpolación `%(...)s` para construir rutas. Leerlo con `configparser`, usar `getint`, `getfloat` y `getboolean`, y `fallback` para las claves que faltan. Después, repetir la interpolación con `ExtendedInterpolation` (`${seccion:clave}`), añadir una sección nueva y guardar el resultado.

In [8]:
import configparser

Path("tiendas.ini").write_text("""\
[DEFAULT]
carpeta = datos
igic = 7.0
abierta_domingo = no

[santa_cruz]
empleados = 12
inventario = %(carpeta)s/santa_cruz.csv
abierta_domingo = yes

[la_laguna]
empleados = 8
inventario = %(carpeta)s/la_laguna.csv
igic = 3.0
""", encoding="utf-8")

ini = configparser.ConfigParser()
ini.read("tiendas.ini", encoding="utf-8")
print(f"{'tienda':<11} {'empl.':>5} {'IGIC':>5} {'domingo':>8}  inventario")
for tienda in ini.sections():
    s = ini[tienda]
    print(f"{tienda:<11} {s.getint('empleados'):>5} {s.getfloat('igic'):>5} {str(s.getboolean('abierta_domingo')):>8}  {s['inventario']}")
print("Gerente (con fallback):", ini.get("la_laguna", "gerente", fallback="sin asignar"))
assert ini["la_laguna"]["inventario"] == "datos/la_laguna.csv"
assert ini.getfloat("santa_cruz", "igic") == 7.0 and not ini.getboolean("la_laguna", "abierta_domingo")

tienda      empl.  IGIC  domingo  inventario
santa_cruz     12   7.0     True  datos/santa_cruz.csv
la_laguna       8   3.0    False  datos/la_laguna.csv
Gerente (con fallback): sin asignar


In [9]:
extendido = configparser.ConfigParser(interpolation=configparser.ExtendedInterpolation())
extendido.read_string("""\
[rutas]
base = /srv/tiendas
copias = ${base}/copias

[arucas]
inventario = ${rutas:base}/arucas.csv
copia = ${rutas:copias}/arucas.bak
""")
print(dict(extendido["arucas"]))
assert extendido["arucas"]["copia"] == "/srv/tiendas/copias/arucas.bak"

ini["telde"] = {"empleados": "5", "inventario": "%(carpeta)s/telde.csv"}
with open("tiendas.ini", "w", encoding="utf-8") as f:
    ini.write(f)
releido = configparser.ConfigParser()
releido.read("tiendas.ini", encoding="utf-8")
print("Secciones guardadas:", releido.sections(), "| inventario de telde:", releido["telde"]["inventario"])
assert releido["telde"]["inventario"] == "datos/telde.csv"

{'inventario': '/srv/tiendas/arucas.csv', 'copia': '/srv/tiendas/copias/arucas.bak'}
Secciones guardadas: ['santa_cruz', 'la_laguna', 'telde'] | inventario de telde: datos/telde.csv


## 9. Buscar en un fichero con `mmap`

`mmap` proyecta un fichero en memoria: se puede indexar y buscar en él como en un `bytes`, pero el sistema operativo solo carga las páginas que se tocan. Contar cuántas medidas de `Izaña` superan 15 °C buscando con `re.finditer` directamente sobre el `mmap`, localizar con `find` y `rfind` la primera y la última aparición de `Hierro` y extraer la línea completa de cada una. Comprobar contra una lectura normal del fichero.

In [10]:
import mmap
import re

patron = re.compile(r"^[^;]+;Izaña;(-?\d+\.\d)".encode(), re.MULTILINE)
nombre = "Hierro".encode()

with open("medidas.csv", "rb") as f, mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as m:
    print(f"Tamaño proyectado: {len(m):,} bytes")
    calurosas = sum(1 for coincidencia in patron.finditer(m) if float(coincidencia[1]) > 15)
    primera, ultima = m.find(nombre), m.rfind(nombre)

    def linea_en(posicion):
        inicio = m.rfind(b"\n", 0, posicion) + 1
        return m[inicio:m.find(b"\n", posicion)].decode()

    extremos = linea_en(primera), linea_en(ultima)

print("Medidas de Izaña por encima de 15 °C:", calurosas)
print("Primera de Hierro:", extremos[0])
print("Última de Hierro: ", extremos[1])
lineas_hierro = [l for l in TEXTO.read_text(encoding="utf-8").splitlines() if ";Hierro;" in l]
assert calurosas == sum(1 for l in lineas if l.split(";")[1] == "Izaña" and float(l.split(";")[2]) > 15)
assert extremos == (lineas_hierro[0], lineas_hierro[-1])

Tamaño proyectado: 2,073,896 bytes
Medidas de Izaña por encima de 15 °C: 304
Primera de Hierro: 2024-01-01T00:00;Hierro;24.1;82
Última de Hierro:  2024-03-10T10:10;Hierro;19.1;39


## 10. Leer por bloques con un generador

Escribir el generador `bloques(ruta, tamaño)` que devuelva el fichero en trozos de bytes con `iter(partial(f.read, tamaño), b"")`, y usarlo para calcular el tamaño y el SHA-256 de `medidas.csv` y de `medidas.csv.xz` sin leerlos enteros, contando además las líneas del CSV (en el comprimido los bytes `
` no son líneas). Comprobar que el resultado no depende del tamaño del bloque y coincide con `hashlib.file_digest`.

In [11]:
import hashlib
from functools import partial


def bloques(ruta, tamaño=64 * 1024):
    with open(ruta, "rb") as f:
        yield from iter(partial(f.read, tamaño), b"")


def resumen(ruta, tamaño=64 * 1024):
    sha, lineas, total = hashlib.sha256(), 0, 0
    for bloque in bloques(ruta, tamaño):
        sha.update(bloque)
        lineas += bloque.count(b"\n")
        total += len(bloque)
    return total, lineas, sha.hexdigest()


for ruta in ["medidas.csv", "medidas.csv.xz"]:
    total, n, huella = resumen(ruta)
    lineas_txt = f"{n} líneas" if ruta.endswith(".csv") else "binario"
    print(f"{ruta:<15} {total:>10,} bytes {lineas_txt:>13}  sha256 {huella[:16]}…")
    assert all(resumen(ruta, t) == (total, n, huella) for t in [1000, 4096, 1 << 20])
    with open(ruta, "rb") as f:
        assert hashlib.file_digest(f, "sha256").hexdigest() == huella

medidas.csv      2,073,896 bytes  60000 líneas  sha256 16638f4d8a7eb53a…
medidas.csv.xz     249,084 bytes       binario  sha256 83b71913eac6283c…


## 11. Una tubería de generadores por líneas

Encadenar generadores para procesar `medidas.csv.gz` sin listas intermedias: `leer_lineas` (abre y produce líneas), `analizar` (las convierte en tuplas), `filtrar_mes` (se queda con un mes) y, al final, un agregado con la máxima temperatura de cada estación en enero. Comprobar que, como los generadores son perezosos, `itertools.islice` puede tomar las tres primeras medidas sin leer el resto del fichero.

In [12]:
from itertools import islice


def leer_lineas(ruta):
    with gzip.open(ruta, "rt", encoding="utf-8") as f:
        yield from f


def analizar(lineas):
    for linea in lineas:
        fecha, estacion, temperatura, humedad = linea.rstrip("\n").split(";")
        yield datetime.fromisoformat(fecha), estacion, float(temperatura), int(humedad)


def filtrar_mes(medidas, mes):
    return (m for m in medidas if m[0].month == mes)


maximas = {}
for _, estacion, temperatura, _ in filtrar_mes(analizar(leer_lineas("medidas.csv.gz")), 1):
    maximas[estacion] = max(maximas.get(estacion, temperatura), temperatura)
for estacion in sorted(maximas):
    print(f"{estacion:<15} {maximas[estacion]:5.1f} °C")

leidas = 0


def contar(lineas):
    global leidas
    for linea in lineas:
        leidas += 1
        yield linea


primeras = list(islice(analizar(contar(leer_lineas("medidas.csv.gz"))), 3))
print("Primeras medidas:", [(m[1], m[2]) for m in primeras], "| líneas leídas:", leidas)
assert leidas == 3
esperado = {}
for l in lineas:
    f, e, t, _ = l.split(";")
    if f[5:7] == "01":
        esperado[e] = max(esperado.get(e, float(t)), float(t))
assert maximas == esperado

Fuerteventura    31.6 °C
Gando            32.0 °C
Hierro           31.0 °C
Izaña            21.8 °C
La Palma         32.7 °C
Tenerife Norte   30.6 °C
Primeras medidas: [('Fuerteventura', 22.6), ('Hierro', 24.1), ('Hierro', 20.7)] | líneas leídas: 3


## 12. Ordenación externa: trozos ordenados

Para ordenar un fichero que «no cabe en memoria», se lee por trozos que sí caben, se ordena cada uno y se guarda en un fichero aparte. Generar `enteros.txt` con 300 000 enteros aleatorios (semilla fija) y escribir `crear_trozos(ruta, carpeta, tamaño)`, que lo parta en trozos ordenados de como mucho `tamaño` números leyendo con `islice`. Comprobar que cada trozo está ordenado y que entre todos tienen los mismos números.

In [13]:
rng = random.Random(7)
with open("enteros.txt", "w") as f:
    f.writelines(f"{rng.randint(-10**6, 10**6)}\n" for _ in range(300_000))


def crear_trozos(ruta, carpeta, tamaño):
    rutas = []
    with open(ruta) as f:
        while trozo := [int(linea) for linea in islice(f, tamaño)]:
            trozo.sort()
            destino = Path(carpeta) / f"trozo_{len(rutas):03}.txt"
            destino.write_text("".join(f"{n}\n" for n in trozo))
            rutas.append(destino)
    return rutas


with tempfile.TemporaryDirectory(dir=".", prefix="trozos_") as carpeta:
    trozos = crear_trozos("enteros.txt", carpeta, 25_000)
    contenidos = [[int(l) for l in t.read_text().split()] for t in trozos]
    print("Trozos:", len(trozos), "| tamaños:", sorted({len(c) for c in contenidos}))
    assert all(c == sorted(c) for c in contenidos)
    assert sorted(n for c in contenidos for n in c) == sorted(int(l) for l in Path("enteros.txt").read_text().split())

Trozos: 12 | tamaños: [25000]


## 13. Ordenación externa: mezclar con `heapq.merge`

Completar la ordenación externa: `ordenar_externo(ruta, salida, tamaño)` crea los trozos en un directorio temporal, los abre todos a la vez con un `ExitStack` y los mezcla con `heapq.merge`, que solo mantiene en memoria un número por trozo, escribiendo el resultado en `enteros_ordenados.txt`. Comprobar que coincide con `sorted` y comparar con `tracemalloc` el pico de memoria frente a ordenarlo todo en memoria.

In [14]:
import heapq
import tracemalloc
from contextlib import ExitStack


def ordenar_externo(ruta, salida, tamaño=25_000):
    with tempfile.TemporaryDirectory(dir=".", prefix="trozos_") as carpeta, ExitStack() as pila:
        ficheros = [pila.enter_context(open(t)) for t in crear_trozos(ruta, carpeta, tamaño)]
        flujos = [map(int, f) for f in ficheros]
        with open(salida, "w") as out:
            out.writelines(f"{n}\n" for n in heapq.merge(*flujos))
        return len(ficheros)


def ordenar_en_memoria(ruta, salida):
    with open(ruta) as f:
        numeros = sorted(int(l) for l in f)
    Path(salida).write_text("".join(f"{n}\n" for n in numeros))


picos = {}
for nombre, funcion, salida in [("externa", ordenar_externo, "enteros_ordenados.txt"),
                                ("en memoria", ordenar_en_memoria, "enteros_ordenados_memoria.txt")]:
    tracemalloc.start()
    funcion("enteros.txt", salida)
    picos[nombre] = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

resultado = [int(l) for l in Path("enteros_ordenados.txt").read_text().split()]
original = [int(l) for l in Path("enteros.txt").read_text().split()]
print("Números ordenados:", len(resultado), "| primeros:", resultado[:3], "| últimos:", resultado[-3:])
print("Pico de la externa menor que el de la ordenación en memoria:", picos["externa"] < picos["en memoria"])
print(f"La externa usa menos de la mitad: {picos['externa'] < picos['en memoria'] / 2}")
assert resultado == sorted(original)
assert Path("enteros_ordenados.txt").read_bytes() == Path("enteros_ordenados_memoria.txt").read_bytes()
assert not list(Path(".").glob("trozos_*"))

Números ordenados: 300000 | primeros: [-999996, -999992, -999989] | últimos: [999982, 999983, 999992]
Pico de la externa menor que el de la ordenación en memoria: True
La externa usa menos de la mitad: True
